# Nepal Real Estate Property Valuation - Machine Learning Training

This notebook documents the machine-learning workflow used by the project. It loads the supplied Nepal real-estate dataset, separates input features and the target, preprocesses numerical and categorical variables, trains a Random Forest regression pipeline, evaluates it using R² on a held-out test set, and saves the trained model for the Django application.

**Target:** `price_npr`

**Model:** `RandomForestRegressor`

In [14]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

ModuleNotFoundError: No module named 'numpy'

## 1. Load the Dataset

The project dataset is stored in `dataset/nepal_gharjagga_data.csv`.

In [1]:
DATASET_PATH = 'dataset/nepal_gharjagga_data.csv'
df = pd.read_csv(DATASET_PATH)

print('Dataset shape:', df.shape)
display(df.head())
print('\nMissing values:')
display(df.isnull().sum())

NameError: name 'pd' is not defined

## 2. Define Features and Target

The seven property characteristics are used as predictors and `price_npr` is the target value.

In [ ]:
feature_columns = [
    'location', 'land_aana', 'floors', 'road_feet',
    'road_type', 'rooms', 'bathrooms'
]
target_column = 'price_npr'

X = df[feature_columns]
y = df[target_column]

print('Input features:', feature_columns)
print('Target:', target_column)

## 3. Preprocessing

Numerical variables are standardized using `StandardScaler`. Categorical variables are converted to numerical form using `OneHotEncoder`. `handle_unknown='ignore'` allows the trained pipeline to process an unseen category during prediction.

In [ ]:
numeric_features = ['land_aana', 'floors', 'road_feet', 'rooms', 'bathrooms']
categorical_features = ['location', 'road_type']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

preprocessor

## 4. Train/Test Split

The data is divided into 80% training data and 20% testing data. A fixed random state of 42 is used to make the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 5. Build and Train the Random Forest Pipeline

The preprocessing and regression model are combined into one pipeline so that the same transformations are automatically applied during training and prediction. The model settings match the project's `train_model.py` implementation.

In [ ]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=150,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train, y_train)
print('Model training completed.')

## 6. Evaluate the Model

R² is the primary evaluation metric used by the project's original training script. MAE and RMSE are also calculated here as additional evaluation measures.

In [ ]:
y_pred = model_pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R² Score: {r2:.4f} ({r2 * 100:.2f}%)')
print(f'MAE: Rs. {mae:,.2f}')
print(f'RMSE: Rs. {rmse:,.2f}')

## 7. Example Predictions

This cell demonstrates how the trained pipeline can predict the value of a new property using the same feature names as the Django application.

In [ ]:
sample_property = pd.DataFrame([{
    'location': 'New Baneshwor, Kathmandu',
    'land_aana': 4.0,
    'floors': 2.5,
    'road_feet': 13,
    'road_type': 'Pitch',
    'rooms': 6,
    'bathrooms': 3
}])

sample_prediction = model_pipeline.predict(sample_property)[0]
print(f'Estimated property value: Rs. {sample_prediction:,.0f}')

## 8. Save the Trained Model

The trained pipeline is saved as a Joblib file. The Django application loads this file from `valuation/ml_models/nepal_gharjagga_model.joblib`.

In [ ]:
MODEL_DIR = 'valuation/ml_models'
MODEL_PATH = os.path.join(MODEL_DIR, 'nepal_gharjagga_model.joblib')
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(model_pipeline, MODEL_PATH)
print(f'Model saved to: {MODEL_PATH}')

## Conclusion

The complete machine-learning pipeline has been documented: dataset loading, feature/target selection, numerical and categorical preprocessing, train/test splitting, Random Forest regression, evaluation, sample prediction, and model export. The saved pipeline can be used directly by the Django property valuation application.